# SDialog dependencies

In [ ]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")

    # Installing sdialog
    !git clone https://github.com/qanastek/sdialog.git
    %cd sdialog
    %pip install -e .
    %cd ..
else:
    print("Running in Jupyter Notebook")
    # Little hack to avoid the "OSError: Background processes not supported." error in Jupyter notebooks"
    get_ipython().system = os.system

## Local installation

Create a `.venv` using the root `requirement.txt` file and Python `3.11.14`

In [ ]:
from sdialog import Dialog
from IPython.display import display

# Load an existing dialogue

In order to run the next steps in a fast manner, we will start from an existing dialog generated using previous tutorials:

In [ ]:
path_dialog = "../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog) and not os.path.exists("./demo_dialog_doctor_patient.json"):
    !wget https://raw.githubusercontent.com/qanastek/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

original_dialog = Dialog.from_file(path_dialog)
original_dialog.print()

# Tutorial 18: Task annotator

The key objective of this tutorial is to apply different microphone impulse responses to the audio obtains after the accoustics simulation of the room, allowing you to hear how the dialogue would sound as if recorded on various real-world devices.

In [ ]:
print(original_dialog.get_annotations())

In [ ]:
from sdialog.audio.dialog import AudioDialog

In [ ]:
test_dialog = AudioDialog.from_dialog(original_dialog)
print(test_dialog.get_annotations())

Convert the original dialog into a audio enhanced dialog

In [ ]:
dialog: AudioDialog = original_dialog.to_audio()

In [ ]:
print(dialog.get_annotations())

In [ ]:
import sdialog

In [ ]:
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "XXX="

In [ ]:
sdialog.config.llm(
    "amazon:anthropic.claude-3-5-sonnet-20240620-v1:0", 
    temperature=0.7, 
    max_tokens=512,
    region_name="us-east-1"
)

In [ ]:
from sdialog.annotators import QuestionAnsweringAnnotator

In [ ]:
annotated_qa = QuestionAnsweringAnnotator().annotate(dialog)
print(annotated_qa.get_annotations())

In [ ]:
from sdialog.annotators import SpokenQuestionAnsweringAnnotator

In [ ]:
annotated_sqa = SpokenQuestionAnsweringAnnotator().annotate(annotated_qa)
print(annotated_sqa.get_annotations())

If the QuestionAnswering stage wasn't run before:

In [ ]:
new_dialog: AudioDialog = original_dialog.to_audio(dialog_dir_name="no_question_answering")

In [ ]:
annotated_sqa_missing = SpokenQuestionAnsweringAnnotator().annotate(new_dialog)
print(annotated_sqa_missing.get_annotations())

You can create your own task annotator:

In [ ]:
import logging
from sdialog import Dialog
from pydantic import BaseModel
from typing import Any, Optional
from sdialog.annotators.annotator import Annotator, TaskModality

class TurnCountingAnnotator(Annotator):
    """
    Annotator for counting the number of turns in a dialog.
    """

    def get_modality(self) -> list[TaskModality]:
        return [
            TaskModality.TEXT_TO_TEXT
        ]

    def get_task_name(self) -> str:
        return "turn_counting"

    def get_requirements(self) -> list[Annotator]:
        return []

    def save(self, data: Any, args: dict[str, Any] = {}) -> None:
        import pandas as pd

        if args is None or "save_path" not in args or args["save_path"] is None:
            logging.warning("[TurnCountingAnnotator] No 'save_path' provided, skipping saving")
            return

        df = pd.DataFrame(data)
        df = df[["question", "answer"]]
        df.to_csv(args["save_path"], index=False)

        logging.info(
            "[TurnCountingAnnotator] "
            f"Data saved to {args['save_path']}"
        )

    def annotate(self, dialog: Dialog, args: dict[str, Any] = {}) -> Dialog:

        logging.info("[TurnCountingAnnotator] Annotating dialog for turn counting tasks")

        dialog = self.check_requirements(dialog)

        _annotations = {
            "data": [
                {
                    "question": "How many turns are there in this dialog?",
                    "answer": str(len(dialog.turns))
                }
            ],
            "modality": self._modality
        }

        dialog.add_annotations(self._task_name, _annotations)

        logging.info("[TurnCountingAnnotator] Annotations done!")

        self.save(data=_annotations["data"], args=args)

        return dialog

You can also apply multiple annotators at the same time:

In [ ]:
new_dialog2: AudioDialog = original_dialog.to_audio(dialog_dir_name="no_question_answering2")

In [ ]:
from sdialog.annotators import apply_annotators

In [ ]:
new_dialog2 = apply_annotators(
    new_dialog2,
    [
        (QuestionAnsweringAnnotator(), {"save_path": "./outputs_to_audio/question_answering.csv"}),
        (TurnCountingAnnotator(), {"save_path": "./outputs_to_audio/turn_counting.csv"}),
        (SpokenQuestionAnsweringAnnotator(), {"save_path": "./outputs_to_audio/spoken_question_answering.csv"})
    ]
)
print(new_dialog2.get_annotations())